# Overview

This notebook performs the initial data audit for the **Higher Education Outcomes Analysis** project, using **2024 C1** as the first analytical cohort.

The purpose of this stage is to establish a reliable understanding of the structure, quality, and integration readiness of the project's core datasets before applying transformations, feature engineering, or analytical modeling.

The audit covers three structurally linked datasets:

- **Enrollment (primary fact table):** academic outcome records at the `(course_code, section_number)` level, including enrollment volume and final student status distributions (dropout, insufficient performance, free status, regular completion, and promoted completion).
- **Programs (reference table):** canonical mapping between `program_code` and program descriptors, used to enrich the main analytical table with program-level context.
- **Offering (operational metadata table):** course-section operational attributes for the selected period, including weekday, shift, delivery mode, and campus assignment.

This notebook focuses on:

- schema inspection and type validation,
- duplicate detection based on natural keys,
- missing value profiling,
- categorical cardinality and semantic consistency checks,
- categorical distribution profiling,
- dataset-level key uniqueness validation,
- cross-dataset referential integrity checks,
- merge coverage assessment across source tables.

A specific audit objective is to evaluate the completeness and reliability of operational metadata originally present in the Enrollment dataset and validate the external Offering dataset as its canonical replacement source for downstream analysis.

The public version of this project uses a **deterministically pseudonymized but structurally equivalent dataset**, preserving relational integrity, statistical properties, and analytical fidelity while protecting institutional confidentiality.

In [1]:
from notebook_utils import ensure_repo_root
# 
ensure_repo_root()

WindowsPath('C:/Github/higher-education-outcomes-analysis')

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_utils import load_data

enrollment = load_data("enrollment")
programs = load_data("programs")
offering = load_data("offering")


In [3]:
from src.config.contracts import (
    ENROLLMENT_SCHEMA,
    OFFERING_SCHEMA,
    PROGRAMS_SCHEMA,
    PRIMARY_KEYS,
)
from src.auditing import (
    inspect_schema,
    check_key_uniqueness,
    check_duplicates,
    check_one_to_one_mapping,
    )

## Datasets Schema Inspection

### Enrollment

In [4]:
inspect_schema(enrollment, ENROLLMENT_SCHEMA)

,column,field_role,observed_dtype,expected_dtype,dtype_match,non_null_count,null_pct,n_unique
0,campus,categorical,object,object,True,117,61.386139,12
1,schedule_time,categorical,object,object,True,122,59.735974,22
2,delivery_mode,categorical,object,object,True,122,59.735974,11
3,weekday,categorical,object,object,True,122,59.735974,6
4,shift,categorical,object,object,True,122,59.735974,5
5,course_name,descriptor,object,object,True,303,0.000000,131
6,course_code,identifier,object,object,True,303,0.000000,131
7,total_enrollment,metric,int64,int64,True,303,0.000000,88
8,free_status_count,metric,int64,int64,True,303,0.000000,53
9,regular_completion_count,metric,int64,int64,True,303,0.000000,53


### Offering

In [5]:
inspect_schema(offering, OFFERING_SCHEMA)

,column,field_role,observed_dtype,expected_dtype,dtype_match,non_null_count,null_pct,n_unique
0,course_code,identifier,object,object,True,305,0.0,138
1,schedule_time,categorical,object,object,True,305,0.0,38
2,delivery_mode,categorical,object,object,True,305,0.0,26
3,section,identifier,int64,int64,True,305,0.0,16
4,weekday,categorical,object,object,True,305,0.0,15
5,shift,categorical,object,object,True,305,0.0,6
6,campus,categorical,object,object,True,305,0.0,6
7,workload,metric,float64,int64,False,305,0.0,4


### Programs

In [6]:
inspect_schema(programs, PROGRAMS_SCHEMA)

,column,field_role,observed_dtype,expected_dtype,dtype_match,non_null_count,null_pct,n_unique
0,program_code,foreign_key,object,object,True,13,0.0,13
1,program_name,descriptor,object,object,True,13,0.0,13


## Primary Keys Uniqueness Validation

### Enrollment

In [7]:
check_key_uniqueness(enrollment, PRIMARY_KEYS["enrollment"])

Key is unique: course_code, section


,key_columns,rows,unique_keys,duplicate_rows,is_unique
0,"course_code, section",303,303,0,True


### Offering

In [8]:
check_key_uniqueness(offering, PRIMARY_KEYS["offering"])

Key is unique: course_code, section


,key_columns,rows,unique_keys,duplicate_rows,is_unique
0,"course_code, section",305,305,0,True


### Programs

In [9]:
check_key_uniqueness(programs, PRIMARY_KEYS["programs"])

Key is unique: program_code


,key_columns,rows,unique_keys,duplicate_rows,is_unique
0,program_code,13,13,0,True


## Duplicate Rows Detection

### Full Row Duplicates

Enrollment

In [10]:
check_duplicates(enrollment, enrollment.columns.to_list())

No duplicates found for subset: course_name, section, total_enrollment, dropout_count, insufficient_count, free_status_count, promoted_completion_count, regular_completion_count, course_code, program_code, shift, weekday, schedule_time, delivery_mode, campus


,course_name,section,total_enrollment,dropout_count,insufficient_count,free_status_count,promoted_completion_count,regular_completion_count,course_code,program_code,shift,weekday,schedule_time,delivery_mode,campus


Offering

In [11]:
check_duplicates(offering, offering.columns.to_list())

No duplicates found for subset: course_code, section, workload, shift, weekday, schedule_time, delivery_mode, campus


,course_code,section,workload,shift,weekday,schedule_time,delivery_mode,campus


### Code-Name Consistency

Course Code - Course Name

In [12]:
check_one_to_one_mapping(enrollment, "course_code", "course_name",)

No inconsistencies detected between `course_code` and `course_name`.


,course_code,unique_values


Course Name - Course Code

In [13]:
check_one_to_one_mapping(enrollment, "course_name", "course_code")

No inconsistencies detected between `course_name` and `course_code`.


,course_name,unique_values


Program Code - Program Name

In [14]:
check_one_to_one_mapping(programs, "program_code", "program_name")

No inconsistencies detected between `program_code` and `program_name`.


,program_code,unique_values


Program Name - Program Code

In [15]:
check_one_to_one_mapping(programs, "program_name", "program_code")

No inconsistencies detected between `program_name` and `program_code`.


,program_name,unique_values
